<hr style="height:1rem;">

## 1.0 Imports

In [1]:
import uuid
import numpy as np
import pandas as pd

import re
import json
import os
from pathlib import Path

import icd10

<hr style="height:1rem;">

## 2.0 Path Settings

In [2]:
CURRENT_PATH = os.getcwd()
CURRENT_DATA_PATH = os.path.join(
    CURRENT_PATH,
    'data'
)
CURRENT_RESULTS_PATH = os.path.join(
    CURRENT_PATH,
    'results'
)
NOTEBOOK_PATH = Path.cwd().resolve().parents[0]
NOTEBOOK_EVALUATION_PATH = os.path.join(
    NOTEBOOK_PATH,
    'evaluation'
)
NOTEBOOK_RESULTS_PATH = os.path.join(
    NOTEBOOK_PATH,
    'results'
)
DISSERTATION_PATH = Path.cwd().resolve().parents[1]
DISSERTATION_DATA_PATH = os.path.join(
    DISSERTATION_PATH,
    'data'
)
PROJECT_PATH = Path.cwd().resolve().parents[2]
PROJECT_DATA_PATH = os.path.join(
    PROJECT_PATH,
    'data'
)

print(CURRENT_PATH)
print(CURRENT_DATA_PATH)
print(CURRENT_RESULTS_PATH)
print(NOTEBOOK_PATH)
print(NOTEBOOK_RESULTS_PATH)
print(NOTEBOOK_EVALUATION_PATH)
print(DISSERTATION_PATH)
print(DISSERTATION_DATA_PATH)
print(PROJECT_PATH)
print(PROJECT_DATA_PATH)

D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\06_knowledge_graph
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\06_knowledge_graph\data
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\06_knowledge_graph\results
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\results
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\evaluation
D:\ODU\03 - Dissertation Defense\003_dissertation
D:\ODU\03 - Dissertation Defense\003_dissertation\data
D:\ODU\03 - Dissertation Defense
D:\ODU\03 - Dissertation Defense\data


<hr style="height:1rem;">

## Read Data into DataFrame (build dict lookup for Diagnosis and Procedure Codes)

### Read Training (TRAIN) Data into DataFrame

In [3]:
TRAIN1_FILENAME = 'ICD10_CODES_NOTES_TRAIN_DISCHARGE_SUMMARY.csv'
TRAIN1_FILE = os.path.join(
    DISSERTATION_DATA_PATH,
    'preprocessed',
    'codiesp',
    TRAIN1_FILENAME
)
train1_df = pd.read_csv(TRAIN1_FILE)
train1_df.head(5)

,article_id,text_reference,reference_position,code_type,icd_code,text
0,S0004-06142005000700014-1,chest X-ray,2163 2171,Procedures,BW03ZZZ,We describe the case of a 37-year-old man with...
1,S0004-06142005000700014-1,intramuscular streptomycin,2787 2801;2810 2823,Procedures,3E02329,We describe the case of a 37-year-old man with...
2,S0004-06142005000700014-1,enlarged right testis,1343 1376,Diagnoses,N44.8,We describe the case of a 37-year-old man with...
3,S0004-06142005000700014-1,exposure to Brucella,594 615,Diagnoses,Z20.818,We describe the case of a 37-year-old man with...
4,S0004-06142005000700014-1,edema,1250 1256,Diagnoses,R60.9,We describe the case of a 37-year-old man with...


### Read Validation (DEV) Data into DataFrame

In [4]:
DEV1_FILENAME = 'ICD10_CODES_NOTES_DEV_DISCHARGE_SUMMARY.csv'
DEV1_FILE = os.path.join(
    DISSERTATION_DATA_PATH,
    'preprocessed',
    'codiesp',
    DEV1_FILENAME
)
dev1_df = pd.read_csv(DEV1_FILE)
dev1_df.head(5)

,article_id,text_reference,reference_position,code_type,icd_code,text
0,S0004-06142005000900016-1,right renal ultrasound,307 316;348 361,Procedures,BT41ZZZ,A 29-year-old woman with a history of duodenal...
1,S0004-06142005000900016-1,kidney scintigraphy,739 756,Procedures,CT13,A 29-year-old woman with a history of duodenal...
2,S0004-06142005000900016-1,stricture at the right ureteropelvic junction,540 583,Diagnoses,Q62.11,A 29-year-old woman with a history of duodenal...
3,S0004-06142005000900016-1,pyelocalyceal ectasia,326 347,Diagnoses,N28.89,A 29-year-old woman with a history of duodenal...
4,S0004-06142005000900016-1,urinary tract infections,198 229,Diagnoses,N39.0,A 29-year-old woman with a history of duodenal...


### Read Test (TEST) Data into DataFrame

In [5]:
TEST1_FILENAME = 'ICD10_CODES_NOTES_TEST_DISCHARGE_SUMMARY.csv'
TEST1_FILE = os.path.join(
    DISSERTATION_DATA_PATH,
    'preprocessed',
    'codiesp',
    TEST1_FILENAME
)
test1_df = pd.read_csv(TEST1_FILE)
test1_df.head(5)

,article_id,text_reference,reference_position,code_type,icd_code,text
0,S0004-06142005000500011-1,sacks,182 190,Diagnoses,S22.49XA,"Patient of 70 years old, retired miner, withou..."
1,S0004-06142005000500011-1,bladder resection,1847 1856;1943 1949,Procedures,0TTB,"Patient of 70 years old, retired miner, withou..."
2,S0004-06142005000500011-1,prostate ultrasound,1204 1213;1351 1359,Procedures,BV49ZZZ,"Patient of 70 years old, retired miner, withou..."
3,S0004-06142005000500011-1,simple cortical cysts in right kidney,1260 1303,Diagnoses,N28.1,"Patient of 70 years old, retired miner, withou..."
4,S0004-06142005000500011-1,disease,207 217,Diagnoses,R69,"Patient of 70 years old, retired miner, withou..."


In [6]:
frames1 = [train1_df, dev1_df, test1_df]
df1 = pd.concat(frames1)
print(len(df1))
df1.head(5)

18435


,article_id,text_reference,reference_position,code_type,icd_code,text
0,S0004-06142005000700014-1,chest X-ray,2163 2171,Procedures,BW03ZZZ,We describe the case of a 37-year-old man with...
1,S0004-06142005000700014-1,intramuscular streptomycin,2787 2801;2810 2823,Procedures,3E02329,We describe the case of a 37-year-old man with...
2,S0004-06142005000700014-1,enlarged right testis,1343 1376,Diagnoses,N44.8,We describe the case of a 37-year-old man with...
3,S0004-06142005000700014-1,exposure to Brucella,594 615,Diagnoses,Z20.818,We describe the case of a 37-year-old man with...
4,S0004-06142005000700014-1,edema,1250 1256,Diagnoses,R60.9,We describe the case of a 37-year-old man with...


In [7]:
# Initialize empty sets
procedure_icd_codes = set()
diagnosis_icd_codes = set()

# Loop through DataFrame and add to respective sets
for index, row in df1.iterrows():
    if row["code_type"] == "Procedures":
        procedure_icd_codes.add(row["icd_code"])
    elif row["code_type"] == "Diagnoses":
        diagnosis_icd_codes.add(row["icd_code"])

# Print the set counts
print("Procedure ICD Codes:", len(procedure_icd_codes))
print("Diagnosis ICD Codes:", len(diagnosis_icd_codes))

Procedure ICD Codes: 870
Diagnosis ICD Codes: 2557


<hr style="height:1rem;">

## Read Data into DataFrame

### Read Training (TRAIN) Data into DataFrame

In [8]:
TRAIN_FILENAME = 'ICD10_CODES_NOTES_TRAIN.csv'
TRAIN_FILE = os.path.join(
    DISSERTATION_DATA_PATH,
    'preprocessed',
    'codiesp',
    TRAIN_FILENAME
)
train_df = pd.read_csv(TRAIN_FILE)
train_df.head(5)

,article_id,text_references,icd_codes,text
0,S2340-98942015000100005-1,metastatic liver lesion; vomiting; disease; re...,C78.7 R11.10 R69 0DTP 0FB0 R06.00 C56.2 R97.1 R55,We report the case of a 62-year-old patient wh...
1,S2254-28842014000200009-1,anemia; deliveries; Caesarean section; tubal l...,D64.9 10E0XZZ 10D00Z1 0UL7 R53.1 R63.0 R63.4 K...,A 33-year-old woman presented to the emergency...
2,S2254-28842012000300010-1,renal insufficiency; ischemic colitis; infecti...,N19 K55.9 B99.9 B96.20 K63.5 I82.210 R57.0 R58...,A 72-year-old male in HD program from December...
3,S1889-836X2016000100006-1,osteoporosis; otosclerosis; elbow fractures; s...,M81.0 H80.90 S42.409 S42.90X Z87.81 R69 S43.40...,A 65-year-old male patient diagnosed with OI f...
4,S1889-836X2015000200005-1,MRI feet; x-ray of feet; MRI of feet; feet ult...,BQ3M BQ0MZZZ BQ3L BH48ZZZ M84.30X R52 R60.9 BQ...,We describe the case of a 47-year-old female p...


### Read Validation (DEV) Data into DataFrame

In [9]:
DEV_FILENAME = 'ICD10_CODES_NOTES_DEV.csv'
DEV_FILE = os.path.join(
    DISSERTATION_DATA_PATH,
    'preprocessed',
    'codiesp',
    DEV_FILENAME
)
dev_df = pd.read_csv(DEV_FILE)
dev_df.head(5)

,article_id,text_references,icd_codes,text
0,S2254-28842013000300009-1,dialysis; depression; gallbladder adenomyoma; ...,5A1D F32.9 D13.5 E27.40 Z99.2 3E1M39Z J45.909 ...,A 73-year-old woman with a history of hyperten...
1,S1889-836X2015000400006-1,fanconi syndrome; hypophosphatemia; muscular w...,E72.09 E83.39 M62.81 R52 R80.9 R53.1 M83.9 B20...,We report the case of a 64-year-old male with ...
2,S1889-836X2015000200005-2,pain; edema; obese; hysterectomy history; left...,R52 R60.9 E66.9 Z90.710 S92.322 BQ0MZZZ D25.9 ...,We describe the case of a 58-year-old woman.\n...
3,S1888-75462014000200009-1,blocking; tachycardia; BRD; ECG; abnormal path...,3E0T3CZ R00.0 I45.10 4A02X4Z 02583ZZ I45.6 I49...,"This is a 16-year-old adolescent, basketball p..."
4,S1887-85712015000200005-1,femoral shaft fracture; left femur shaft fract...,S72.302 W40.9XX 0QS9 2W37X1Z,A five-year-old male patient who after sufferi...


### Read Test (TEST) Data into DataFrame

In [10]:
TEST_FILENAME = 'ICD10_CODES_NOTES_TEST.csv'
TEST_FILE = os.path.join(
    DISSERTATION_DATA_PATH,
    'preprocessed',
    'codiesp',
    TEST_FILENAME
)
test_df = pd.read_csv(TEST_FILE)
test_df.head(5)

,article_id,text_references,icd_codes,text
0,S2254-28842014000300010-1,Chronic renal failure secondary to nephroangio...,N18.9 I99.8 BR39 R53.81 I21.3 M46.40 M54.5 R50...,A 57-year-old male with a history of arterial ...
1,S1888-75462017000100042-1,hypesthesia; paresthesia; pain; human ultrasou...,R20.1 R20.2 R52 BP4N BP2L R60.9 M79.603 G62.9 ...,A 34-year-old male presented with a two-week h...
2,S1888-75462016000400180-1,thrombosed veins; nerve compression; venous th...,I82.90 G58.9 I82.619 S42.009 4A02X4Z S42.021 J...,"A 53-year-old male, road veteran cyclist, with..."
3,S1887-85712017000100004-1,right internal fixation by femoral intramedull...,0QSC36Z S72.401C BQ27 BQ23 W32.0XX M25.60 R52 ...,"22-year-old man weight, 172 cm.), active milit..."
4,S1887-85712016000100005-1,hematoma; acute compartment syndrome in right ...,T14.8 M79.A11 M15.9 R07.9 R52 R58 I10 M19.90 C...,The case described in our study is that of a 7...


### Concatenate the 3 DataFrames into 1 DataFrame

In [11]:
frames = [train_df, dev_df, test_df]
df = pd.concat(frames)
print(len(df))
df.head(5)

1000


,article_id,text_references,icd_codes,text
0,S2340-98942015000100005-1,metastatic liver lesion; vomiting; disease; re...,C78.7 R11.10 R69 0DTP 0FB0 R06.00 C56.2 R97.1 R55,We report the case of a 62-year-old patient wh...
1,S2254-28842014000200009-1,anemia; deliveries; Caesarean section; tubal l...,D64.9 10E0XZZ 10D00Z1 0UL7 R53.1 R63.0 R63.4 K...,A 33-year-old woman presented to the emergency...
2,S2254-28842012000300010-1,renal insufficiency; ischemic colitis; infecti...,N19 K55.9 B99.9 B96.20 K63.5 I82.210 R57.0 R58...,A 72-year-old male in HD program from December...
3,S1889-836X2016000100006-1,osteoporosis; otosclerosis; elbow fractures; s...,M81.0 H80.90 S42.409 S42.90X Z87.81 R69 S43.40...,A 65-year-old male patient diagnosed with OI f...
4,S1889-836X2015000200005-1,MRI feet; x-ray of feet; MRI of feet; feet ult...,BQ3M BQ0MZZZ BQ3L BH48ZZZ M84.30X R52 R60.9 BQ...,We describe the case of a 47-year-old female p...


<hr style="height:1rem;">

## Get ICD Code Titles

In [12]:
TEMPLATE_PREFIX = """You are an expert in medical NLP and knowledge graph construction. Your task is to analyze clinical notes and extract relationships between ICD codes to build a structured knowledge graph.

### **Input Data:**
"""
print(TEMPLATE_PREFIX)

You are an expert in medical NLP and knowledge graph construction. Your task is to analyze clinical notes and extract relationships between ICD codes to build a structured knowledge graph.

### **Input Data:**



In [13]:
TEMPLATE_SUFFIX = """### **Extraction Guidelines:**
1. **Entity Recognition:**
   - Identify diseases, symptoms, and medical conditions from the clinical notes.
   - Standardize terms using **ICD-10, SNOMED CT, or UMLS ontology**.

2. **Relationship Extraction:**
   - Determine the relationship type between {CONDITION_NAME_1} and {CONDITION_NAME_2}.
   - Use predefined medical relationships such as:
     - **Causal (CAUSES)**
     - **Complication of (COMPLICATES)**
     - **Risk Factor for (INCREASES_RISK)**
     - **Associated with (ASSOCIATED_WITH)**
     - **No direct relationship (NO_RELATIONSHIP)**

3. **Graph Representation:**
   - Output the extracted information in a structured **triplet format** for knowledge graph integration.
   - Format: **(Entity 1) —[Relationship]→ (Entity 2)**

### **Output Format (JSON for KG Construction):**
```json
{
  "nodes": [
    {"id": "{ICD_CODE_1}", "name": "{CONDITION_NAME_1}", "type": "Disease"},
    {"id": "{ICD_CODE_2}", "name": "{CONDITION_NAME_2}", "type": "Disease"}
  ],
  "edges": [
    {
      "source": "{ICD_CODE_1}",
      "target": "{ICD_CODE_2}",
      "relationship": "{RELATIONSHIP_TYPE}",
      "confidence": "{CONFIDENCE_LEVEL}",
      "supporting_text": "{EVIDENCE_FROM_NOTES}"
    }
  ]
}
"""
print(TEMPLATE_SUFFIX)

### **Extraction Guidelines:**
1. **Entity Recognition:**
   - Identify diseases, symptoms, and medical conditions from the clinical notes.
   - Standardize terms using **ICD-10, SNOMED CT, or UMLS ontology**.

2. **Relationship Extraction:**
   - Determine the relationship type between {CONDITION_NAME_1} and {CONDITION_NAME_2}.
   - Use predefined medical relationships such as:
     - **Causal (CAUSES)**
     - **Complication of (COMPLICATES)**
     - **Risk Factor for (INCREASES_RISK)**
     - **Associated with (ASSOCIATED_WITH)**
     - **No direct relationship (NO_RELATIONSHIP)**

3. **Graph Representation:**
   - Output the extracted information in a structured **triplet format** for knowledge graph integration.
   - Format: **(Entity 1) —[Relationship]→ (Entity 2)**

### **Output Format (JSON for KG Construction):**
```json
{
  "nodes": [
    {"id": "{ICD_CODE_1}", "name": "{CONDITION_NAME_1}", "type": "Disease"},
    {"id": "{ICD_CODE_2}", "name": "{CONDITION_NAME_2}", "type": "

In [14]:
def format_icd10_codes(icd_codes, diagnosis_icd_codes):
    """
    Formats ICD-10 codes with their corresponding condition names.

    :param icd_codes: List of ICD-10 codes (e.g., ['E11.00', 'N18.9'])
    :param icd_mapping: Dictionary mapping ICD-10 codes to condition names
                        (e.g., {'E11.00': 'Type 2 diabetes mellitus', 'N18.9': 'Chronic kidney disease'})
    :return: Formatted string with the structured ICD-10 code information
    """
    output = []
    count = 1
    is_icd_code = False
    for index, code in enumerate(icd_codes, start=1):
        if code in diagnosis_icd_codes:
            c = icd10.find(code)
            if c:
                condition_name = c.description
                output.append(f"- **ICD Code {count}:** {code} ({condition_name})")
                count +=1
                is_icd_code = True
    return is_icd_code, "\n".join(output)

In [15]:
# Loop through DataFrame and add to respective sets
prompt_template_list = []
is_icd_code_found_list = []
for index, row in df.iterrows():
    codes_str = str(row["icd_codes"])
    text_str = str(row["text"])

    icd_codes = codes_str.split()
    clinical_note = "- **Clinical Notes:** " + text_str.replace("\n", " ")
    
    is_icd_code, formatted_icd_code_output = format_icd10_codes(icd_codes, diagnosis_icd_codes) 

    prompt_template = str(TEMPLATE_PREFIX + formatted_icd_code_output + "\n" + clinical_note + "\n" + TEMPLATE_SUFFIX)
    prompt_template_list.append(prompt_template)
    is_icd_code_found_list.append(is_icd_code)

print(len(prompt_template_list))
print(len(is_icd_code_found_list))

df['prompt_template'] = prompt_template_list
df['is_icd_in_template'] = is_icd_code_found_list

1000
1000


In [16]:
df.head(5)

,article_id,text_references,icd_codes,text,prompt_template,is_icd_in_template
0,S2340-98942015000100005-1,metastatic liver lesion; vomiting; disease; re...,C78.7 R11.10 R69 0DTP 0FB0 R06.00 C56.2 R97.1 R55,We report the case of a 62-year-old patient wh...,You are an expert in medical NLP and knowledge...,True
1,S2254-28842014000200009-1,anemia; deliveries; Caesarean section; tubal l...,D64.9 10E0XZZ 10D00Z1 0UL7 R53.1 R63.0 R63.4 K...,A 33-year-old woman presented to the emergency...,You are an expert in medical NLP and knowledge...,True
2,S2254-28842012000300010-1,renal insufficiency; ischemic colitis; infecti...,N19 K55.9 B99.9 B96.20 K63.5 I82.210 R57.0 R58...,A 72-year-old male in HD program from December...,You are an expert in medical NLP and knowledge...,True
3,S1889-836X2016000100006-1,osteoporosis; otosclerosis; elbow fractures; s...,M81.0 H80.90 S42.409 S42.90X Z87.81 R69 S43.40...,A 65-year-old male patient diagnosed with OI f...,You are an expert in medical NLP and knowledge...,True
4,S1889-836X2015000200005-1,MRI feet; x-ray of feet; MRI of feet; feet ult...,BQ3M BQ0MZZZ BQ3L BH48ZZZ M84.30X R52 R60.9 BQ...,We describe the case of a 47-year-old female p...,You are an expert in medical NLP and knowledge...,True


In [17]:
OUTPUT_FILENAME = 'ICD10_CODES_NOTES_1000.csv'
OUTPUT_FILE = os.path.join(
    CURRENT_DATA_PATH,
    OUTPUT_FILENAME
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

In [18]:
print(len(df))

1000


In [19]:
print(df.iloc[0]['prompt_template'])

You are an expert in medical NLP and knowledge graph construction. Your task is to analyze clinical notes and extract relationships between ICD codes to build a structured knowledge graph.

### **Input Data:**
- **ICD Code 1:** C78.7 (Secondary malignant neoplasm of liver and intrahepatic bile duct)
- **ICD Code 2:** R11.10 (Vomiting, unspecified)
- **ICD Code 3:** R69 (Illness, unspecified)
- **ICD Code 4:** R06.00 (Dyspnea, unspecified)
- **ICD Code 5:** C56.2 (Malignant neoplasm of left ovary)
- **ICD Code 6:** R97.1 (Elevated cancer antigen 125 [CA 125])
- **ICD Code 7:** R55 (Syncope and collapse)
- **Clinical Notes:** We report the case of a 62-year-old patient who in December 2010 was operated on for a serous papillary carcinoma in the left ovary with involvement of a ganglia of the chain referral (stage IIIC). Subsequently, she received adjuvant chemotherapy with ropivacaine and paclitaxel, finalized in June 2011. Later, asymptomatic follow-up began until April 2013, when an in